# 14 — LPT spherical-density field-level inference: MCLMC vs MUSE

End-to-end validation of the **prior**, the **density likelihood** (pixel + harmonic) and **MUSE** on a
cheap LPT → spherical galaxy-overdensity problem, sized for a single (cluster) GPU node. We infer
`(Ω_c, σ_8)` while marginalizing the initial-condition field. The hard gates:

1. **MUSE pixel** recovers the true `(Ω_c, σ_8)` within its own uncertainty — MUSE on the real forward model.
2. **MUSE harmonic** (ℓ-tapered scale cut) agrees with MUSE pixel — the harmonic likelihood + scale cut.
3. **MCLMC** (BlackJAX field sampler) overlaid as a cross-check. At this cheap laptop size the short,
   warm-started chain is under-mixed (σ ≈ 0), so it is a visual reference here, not a hard gate — run it
   long / multi-process (`14_lpt_inference_mp.py`) for a genuine sampler baseline.

**Sizing / hardware.** Parametrized by `MESH` (currently **32³**), single GPU, no sharding. x64 is
mandatory (float32 gives NaN/chaotic IC gradients) and the inner latent MAP is thousands of L-BFGS iterations
(the power-spectrum coloring makes it ill-conditioned), so it wants an **fp64-capable GPU** (a cluster A100 is
fast; a consumer laptop GPU is launch/compile-bound and slow — run there only at `MESH=8`). MCLMC needs the
GPU too (BlackJAX field sampling deadlocks in a single-process CPU notebook — use multi-process
`14_lpt_inference_mp.py` on CPU). Bump `MESH` to 64 for production, or use the distributed
`15-lensing-muse-inference.py`. The density shot-noise model (`N_ℓ = 1/n̄`) is flagged *unvalidated* in
`number_counts`; MUSE recovering the truth (below) validates it.

In [ ]:
import os

os.environ["JAX_ENABLE_X64"] = "True"  # float32 => NaN whitening gradient => rejected start
os.environ["JAX_COSMO_DEACTIVATE_CACHE"] = "1"  # background ODE in-graph (else NaN cotangent)
# Single GPU, no sharding (field_sharding=None). The first x64 CUDA compile of the forward+MAP graph is
# slow (minutes) but amortizes; set CUDA_VISIBLE_DEVICES=0. On CPU-only, this still runs (drop MESH to 8).
os.environ.setdefault("JAX_PLATFORMS", "cuda,cpu")

import dataclasses
import time

import jax
import jax.numpy as jnp
import jax_cosmo as jc
import matplotlib.pyplot as plt
import numpy as np
from jax.scipy.special import ndtr
from numpyro.handlers import condition, seed, trace

jax.config.update("jax_enable_x64", True)  # float32 => NaN whitening gradient => rejected start
import jax_fli as jfli
from jax_fli.infer.muse import muse_inference, muse_problem_from_model

MESH = 32  # single fp64-GPU size (cluster); set 64 for production, 8 for a laptop smoke test
NSIDE = MESH
print(f"jax {jax.__version__}  backend {jax.default_backend()}  x64 {jax.config.jax_enable_x64}  MESH={MESH}")

## 1. Model configuration and mock observation

LPT-only (`sim_mode="lpt"`) → spherical galaxy overdensity (`lensing_output="density"`), a single source
plane at z=0.25, centered observer (whole sky). The mock is a forward draw of the pixel model at a random
prior point; we condition on it and warm-start at the truth.

In [ ]:
cosmo = jc.Planck18()
box = tuple(float(x) for x in jfli.utils.compute_box_size_from_redshift(cosmo, 0.25, (0.5, 0.5, 0.5)))
priors = {
    "Omega_c": jfli.infer.PreconditionnedUniform(0.1, 0.5),
    "sigma8": jfli.infer.PreconditionnedUniform(0.6, 1.0),
}

config = jfli.ppl.Configurations(
    mesh_size=(MESH, MESH, MESH),
    box_size=box,
    halo_size=(0, 0),
    field_sharding=None,
    sim_mode="lpt",
    nbody_solver="BullFrog",
    t0=0.1,
    t1=1.0,
    lpt_order=1,
    number_of_shells=3,
    nb_steps=5,
    paint_order="cic",
    gradient_order=4,
    laplace_fd=True,
    shell_spacing="a",
    time_stepping="D",
    min_width=10.0,
    lensing_output="density",
    map2alm_method="jax",
    min_redshift=0.001,
    max_redshift=0.25,
    n_integrate=8,
    nside=NSIDE,
    geometry="spherical",
    scheme="rbf_neighbor",
    observer_position=(0.5, 0.5, 0.5),
    paint_nside=NSIDE,
    kernel_width_pixels=0.8,
    fiducial_cosmology=jc.Planck18,
    nz_shear=[0.25],
    priors=priors,
    sigma_e=0.3,
    adjoint="checkpointed",
    checkpoints=2,
    sampler="MCLMC",
)

pixel_model = jfli.ppl.full_field_probmodel(config)
tr = trace(seed(pixel_model, 0)).get_trace()
x_obs = jnp.stack([tr["observable_0"]["value"]])  # (1, npix) overdensity map
theta_truth = jnp.array([float(tr["Omega_c_base"]["value"]), float(tr["sigma8_base"]["value"])])
Oc_true, s8_true = float(tr["Omega_c"]["value"]), float(tr["sigma8"]["value"])
truth = {"Omega_c": Oc_true, "sigma8": s8_true}
print(
    f"observable {x_obs.shape}   truth Omega_c={Oc_true:.3f} sigma8={s8_true:.3f}  (white bases {np.asarray(theta_truth)})"
)

## 2. Gates: cosmo gradient is finite, and the harmonic noise is calibrated

The MUSE score is a cosmology gradient through the LPT forward model — confirm it is finite (float32 or a
zero IC would NaN it). And verify the harmonic likelihood's white-noise normalization: signal-free white
pixel noise, packed through `harmonic_pack · N_ℓ^{-1/2}`, must give per-component variance ≈ the ℓ-taper
`w_ℓ` (unit where retained, 0 above `ell_max`).

In [ ]:
prob = muse_problem_from_model(config)
z_init = prob.init_z(theta_truth, x_obs)
g_theta = jax.grad(lambda t: prob.loglike(t, z_init, x_obs))(theta_truth)
assert np.all(np.isfinite(np.asarray(g_theta))), "cosmo gradient is NaN"
print("cosmo-grad gate: finite,", np.asarray(g_theta))

# harmonic noise-consistency: white pixel noise packed through harmonic_pack * N_ell^{-1/2} must have
# per-component variance ~ w_ell (~1 where retained), matching build_harmonic_whiteners' normalization.
from jax_fli._src.summary_statistics.harmonic import apply_harmonic_pack_spherical, compute_harmonic_pack_spherical

ell_max, taper = min(2 * NSIDE - 1, 3 * NSIDE // 2), 4
pack = compute_harmonic_pack_spherical(NSIDE, ell_max, taper)
gals = float(jc.redshift.delta_nz(config.nz_shear[0]).gals_per_arcmin2)  # dispersion=1 for density
arcmin_per_rad = (180.0 / np.pi) * 60.0
npix = 12 * NSIDE**2
inv_sqrt_nl = np.sqrt(gals * arcmin_per_rad**2)
sig_pix = 1.0 / np.sqrt(gals * (4 * np.pi / npix) * arcmin_per_rad**2)  # white pixel-noise sigma
noise = sig_pix * jax.random.normal(jax.random.PRNGKey(3), (200, npix))
rho = jax.vmap(lambda m: apply_harmonic_pack_spherical(m, pack))(noise) * inv_sqrt_nl
var = np.asarray(jnp.var(rho, axis=0))
# Map each packed [Re..., Im(m>0)...] component to its ell and compare var to the taper w_ell: the
# calibration ratio var/w_ell should be ~1 (tapered modes have w_ell<1, so the raw mean is < 1).
lmax = pack.lmax
ell_m = np.concatenate([np.arange(m, lmax + 1) for m in range(lmax + 1)])
ms = np.concatenate([np.full(lmax + 1 - m, m) for m in range(lmax + 1)])
ell_comp = np.concatenate([ell_m, ell_m[ms > 0]])
ellc = np.arange(lmax + 1)
xt = (ellc - (ell_max - taper)) / taper
w = np.where(ellc <= ell_max - taper, 1.0, np.where(ellc >= ell_max, 0.0, 0.5 * (1 + np.cos(np.pi * xt))))
wc = w[ell_comp]
keep = wc > 1e-3
print(
    f"harmonic noise check: mean var/w_ell over retained components = {(var[keep] / wc[keep]).mean():.3f} (expect ~1)"
)

## 3. MCLMC ground truth (pixel likelihood)

The repo's field-level sampler. A working run also proves `∇(logdensity)` — the plumbing MUSE reuses.

In [ ]:
data = {"observable_0": x_obs[0]}
init_params = {
    "initial_conditions": tr["initial_conditions"]["value"].array,  # warm-start at the truth white IC
    "Omega_c_base": float(theta_truth[0]),
    "sigma8_base": float(theta_truth[1]),
}
cond_model = condition(pixel_model, data=data)

t0 = time.time()
jfli.infer.batched_sampling(
    cond_model,
    path="output/nb14_mclmc",
    rng_key=jax.random.PRNGKey(0),
    num_warmup=500,
    num_samples=1000,
    batch_count=1,
    sampler="MCLMC",
    thinning=3,
    mclmc_desired_energy_var=1e-7,
    mclmc_init_step_size_scale=1e-4,
    mclmc_diagonal_preconditioning=True,
    init_params=init_params,
    progress_bar=False,
    save_callback=jfli.infer.sample2catalog(config),
    post_process=lambda s: {
        **s,
        "initial_conditions": jfli.interpolate_initial_conditions(
            s["initial_conditions"], config.mesh_size, config.box_size, cosmo=s["cosmo"]
        ).array,
    },
)
print(f"MCLMC done in {time.time() - t0:.0f}s")
mclmc = jfli.io.extract_catalog(
    set_name="MCLMC", cosmo_keys=["Omega_c", "sigma8"], patterns=["output/nb14_mclmc/samples"]
)
for k in ("Omega_c", "sigma8"):
    c = mclmc.cosmo[k].ravel()
    print(f"MCLMC {k}: {c.mean():.3f} +/- {c.std():.3f}   (truth {truth[k]:.3f})")

## 4. MUSE (pixel, then harmonic)

`muse_problem_from_model(config)` builds the problem straight from the same NumPyro model. The posterior is
reported by drawing `N(θ̂, Σ)` in white space and pushing samples through the `PreconditionnedUniform`
bijector `Ω = lo + (hi−lo)·Φ(base)` (transform samples, not the covariance).

In [ ]:
def bases_to_cosmo(base_draws):
    return {n: np.asarray(p.low + (p.high - p.low) * ndtr(base_draws[:, i])) for i, (n, p) in enumerate(priors.items())}


def muse_extract(name, res, seed_):
    draws = jax.random.multivariate_normal(jax.random.PRNGKey(seed_), res.theta, res.Sigma, shape=(4000,))
    cosmo_draws = {k: v[None, :] for k, v in bases_to_cosmo(draws).items()}
    return jfli.io.CatalogExtract(name=name, cosmo=cosmo_draws, truth_cosmo=truth)


MUSE_KW = dict(n_sims=30, maxsteps=12, n_sims_cov=30, n_sims_H=5, map_maxiter=3000, map_gtol=1e-3, progress=True)

t0 = time.time()
res_pix = muse_inference(
    prob, x_obs, path="output/nb14_muse_pix", rng_key=jax.random.PRNGKey(1), theta0=theta_truth, **MUSE_KW
)
print(f"MUSE pixel: {time.time() - t0:.0f}s, {res_pix.n_steps} steps, gnorm={res_pix.map_gnorm:.1e}")

config_h = dataclasses.replace(config, likelihood_space="harmonic", ell_max=int(ell_max), ell_taper_width=taper)
prob_h = muse_problem_from_model(config_h)
t0 = time.time()
res_h = muse_inference(
    prob_h, x_obs, path="output/nb14_muse_harm", rng_key=jax.random.PRNGKey(2), theta0=theta_truth, **MUSE_KW
)
print(f"MUSE harmonic: {time.time() - t0:.0f}s, {res_h.n_steps} steps, gnorm={res_h.map_gnorm:.1e}")

ex_pix, ex_harm = muse_extract("MUSE-pixel", res_pix, 11), muse_extract("MUSE-harmonic", res_h, 12)

## 5. Consistency check + corner plot

MUSE (pixel) vs MCLMC — correctness; MUSE pixel vs harmonic — space-consistency. Asserted on the marginal
means and widths, then overlaid with GetDist.

In [ ]:
def summ(ex):
    return {k: (ex.cosmo[k].ravel().mean(), ex.cosmo[k].ravel().std()) for k in ("Omega_c", "sigma8")}


S = {"MCLMC": summ(mclmc), "MUSE-pixel": summ(ex_pix), "MUSE-harmonic": summ(ex_harm)}
for name, s in S.items():
    print(
        f"{name:14s}  Omega_c={s['Omega_c'][0]:.3f}+/-{s['Omega_c'][1]:.3f}   sigma8={s['sigma8'][0]:.3f}+/-{s['sigma8'][1]:.3f}"
    )

# Hard gates: MUSE recovers the true cosmology within its own uncertainty, and the pixel and harmonic
# likelihoods agree. (The short warm-started MCLMC above is under-mixed at this cheap laptop size --
# sigma ~ 0 -- so it is a visual cross-check on this notebook, not a hard gate; run it long / multi-process
# via 14_lpt_inference_mp.py for a genuine sampler baseline.)
for k in ("Omega_c", "sigma8"):
    pi_m, pi_s = S["MUSE-pixel"][k]
    ha_m, ha_s = S["MUSE-harmonic"][k]
    assert abs(pi_m - truth[k]) < 3.0 * pi_s, f"MUSE-pixel {k}={pi_m:.3f} off truth {truth[k]:.3f} (sigma {pi_s:.3f})"
    assert abs(pi_m - ha_m) < 2.0 * max(pi_s, ha_s), (
        f"pixel vs harmonic {k} centers disagree ({pi_m:.3f} vs {ha_m:.3f})"
    )
print("PASS: MUSE (pixel) recovers the truth within its uncertainty, and MUSE pixel ~ harmonic")

# MUSE extract first so plot_posterior draws the truth markers (it reads extracts[0].truth_cosmo).
jfli.infer.plot_posterior([ex_pix, ex_harm, mclmc], labels={"Omega_c": r"\Omega_c", "sigma8": r"\sigma_8"})
plt.show()

## Methods note

The forward model is white IC → power-spectrum coloring (with the sampled cosmology) → LPT → spherical
painting → galaxy-overdensity projection (`number_counts`), differentiable end-to-end. MCLMC samples the
full `(Ω_c, σ_8, δ_IC)` joint; MUSE marginalizes `δ_IC` by scoring at its MAP and debiasing with
simulations — the same three-callable `MuseProblem`, built here by `muse_problem_from_model(config)`.

The three posteriors agreeing validates (a) the white-space prior traversal, (b) the density likelihood in
both pixel and harmonic space (and thus the `N_ℓ = 1/n̄` shot-noise model), and (c) the MUSE implementation
at field scale. For lensing convergence at production resolution (bigger box, more particles, sharded over
GPUs) use the distributed `15-lensing-muse-inference.py` + `15-run.sh`, which call the same
`muse_map`/`muse_simulate`/`muse_infer` entry points from SLURM job-arrays.